# Electricity Demand Forecasting — Interactive Notebook
Run cells top-to-bottom. Change parameters in the **CONFIG** cell to customise the experiment.

In [ ]:
# ── Install dependencies (run once) ──────────────────────────────────────────
# !pip install -r requirements.txt

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0d0f14'
matplotlib.rcParams['axes.facecolor']   = '#151820'
matplotlib.rcParams['text.color']       = '#e2e4f0'

from utils.data_loader   import load_csv, engineer_features, time_series_split
from utils.metrics       import evaluate, compare_models, print_leaderboard
from utils.visualization import (plot_forecast_vs_actual, plot_residuals,
                                  plot_feature_importance, plot_metrics_comparison)
from main import generate_demo_data

print('Imports OK ✓')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  CONFIG — edit these values
# ════════════════════════════════════════════════════════════════════════════

CSV_PATH     = None          # Set to 'path/to/your/data.csv' or leave None for demo
TEST_SIZE    = 0.20          # Fraction of data used for evaluation
HORIZON      = 24            # Forecast horizon in hours
MODELS       = ['arima', 'prophet', 'xgb', 'lgb', 'rf', 'lstm']  # Which models to run
TUNE_XGB     = False         # Set True to run Optuna hyperparameter search for XGBoost
TUNE_TRIALS  = 50
SAVE_RESULTS = True          # Save leaderboard CSV and plots to results/

## 1 · Load Data

In [ ]:
if CSV_PATH:
    df = load_csv(CSV_PATH)
else:
    print('No CSV_PATH set — using demo data.')
    df = generate_demo_data(n_days=60)

print(f'\nShape: {df.shape}')
print(f'Demand range: {df["demand"].min():.1f} – {df["demand"].max():.1f} MW')
df.head()

In [ ]:
# Quick overview plot
fig, axes = plt.subplots(2, 1, figsize=(14, 6))
df['demand'].plot(ax=axes[0], color='#4a9eff', lw=0.8, title='Full demand series')
df['demand'].resample('D').mean().plot(ax=axes[1], color='#3dd68c', lw=1.5, title='Daily average')
for ax in axes:
    ax.set_ylabel('MW')
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2 · Feature Engineering & Train/Test Split

In [ ]:
fe_df = engineer_features(df)
X_train, y_train, X_test, y_test = time_series_split(fe_df, test_size=TEST_SIZE)

# Raw series for ARIMA / Prophet / LSTM
raw   = df['demand'].values
split = int(len(raw) * (1 - TEST_SIZE))
y_train_raw, y_test_raw = raw[:split], raw[split:]
test_index = df.index[split:]

print(f'Feature columns: {len(X_train.columns)}')
print(X_train.columns.tolist())

## 3 · Fit Models

In [ ]:
results = {}

# ── SARIMA ────────────────────────────────────────────────────────────────
if 'arima' in MODELS:
    from models.sarima_model import SARIMAForecaster
    m = SARIMAForecaster(seasonal_period=24)
    m.fit(y_train_raw)
    preds, ci = m.predict(len(y_test_raw), return_conf_int=True)
    results['SARIMA'] = dict(preds=preds, ci=ci,
                              metrics=evaluate(y_test_raw, preds, 'SARIMA', train_time_s=m.train_time_))

# ── Prophet ───────────────────────────────────────────────────────────────
if 'prophet' in MODELS:
    from models.prophet_model import ProphetForecaster
    extra = [c for c in ['temperature','holiday'] if c in df.columns]
    m = ProphetForecaster(extra_regressors=extra)
    m.fit(df['demand'].iloc[:split],
          regressors_train=df[extra].iloc[:split] if extra else None)
    fc = m.predict(test_index, regressors_future=df[extra].iloc[split:] if extra else None)
    preds = fc['yhat'].values
    results['Prophet'] = dict(preds=preds, ci=fc[['yhat_lower','yhat_upper']].values,
                               metrics=evaluate(y_test_raw, preds, 'Prophet', train_time_s=m.train_time_))

# ── XGBoost ───────────────────────────────────────────────────────────────
if 'xgb' in MODELS:
    from models.tree_models import XGBoostForecaster
    m = XGBoostForecaster()
    if TUNE_XGB:
        m.tune(X_train, y_train, n_trials=TUNE_TRIALS)
    m.fit(X_train, y_train, X_test, y_test)
    preds = m.predict(X_test)
    results['XGBoost'] = dict(preds=preds, importance=m.feature_importances_,
                               metrics=evaluate(y_test.values, preds, 'XGBoost', train_time_s=m.train_time_))

# ── LightGBM ──────────────────────────────────────────────────────────────
if 'lgb' in MODELS:
    from models.tree_models import LightGBMForecaster
    m = LightGBMForecaster()
    m.fit(X_train, y_train, X_test, y_test)
    preds = m.predict(X_test)
    results['LightGBM'] = dict(preds=preds, importance=m.feature_importances_,
                                metrics=evaluate(y_test.values, preds, 'LightGBM', train_time_s=m.train_time_))

# ── Random Forest ─────────────────────────────────────────────────────────
if 'rf' in MODELS:
    from models.tree_models import RandomForestForecaster
    m = RandomForestForecaster()
    m.fit(X_train, y_train)
    preds = m.predict(X_test)
    results['RandomForest'] = dict(preds=preds, importance=m.feature_importances_,
                                    metrics=evaluate(y_test.values, preds, 'RandomForest', train_time_s=m.train_time_))

# ── LSTM ──────────────────────────────────────────────────────────────────
if 'lstm' in MODELS:
    from models.lstm_model import LSTMForecaster
    m = LSTMForecaster(lookback=48, horizon=HORIZON, epochs=50)
    m.fit(y_train_raw)
    preds = []
    for i in range(len(y_test_raw)):
        ctx = raw[max(0, split+i-48): split+i]
        preds.append(m._predict_one(ctx))
    preds = np.array(preds)
    results['LSTM'] = dict(preds=preds,
                            metrics=evaluate(y_test_raw, preds, 'LSTM', train_time_s=m.train_time_),
                            history=m.history_)

print(f'\n✓ Finished fitting {len(results)} models.')

## 4 · Leaderboard

In [ ]:
metrics_dict = {k: v['metrics'] for k, v in results.items()}
print_leaderboard(metrics_dict)
metrics_df = compare_models(metrics_dict)
if SAVE_RESULTS:
    import os; os.makedirs('results', exist_ok=True)
    metrics_df.to_csv('results/leaderboard.csv', index=False)
    print('Saved → results/leaderboard.csv')
metrics_df

## 5 · Forecast vs Actual

In [ ]:
actual_s   = pd.Series(y_test_raw, index=test_index[:len(y_test_raw)])
preds_dict = {k: v['preds'] for k, v in results.items()}
fig = plot_forecast_vs_actual(actual_s, preds_dict, n_display=168)
if SAVE_RESULTS:
    fig.savefig('results/forecast_vs_actual.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

## 6 · MAPE & RMSE Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, metric in zip(axes, ['MAPE', 'RMSE']):
    df_m = metrics_df.sort_values(metric)
    from utils.visualization import _color
    colors = [_color(m) for m in df_m['model']]
    ax.barh(df_m['model'], df_m[metric], color=colors, height=0.55)
    ax.set_title(f'{metric} — lower is better')
    ax.grid(axis='x')
plt.tight_layout()
if SAVE_RESULTS:
    fig.savefig('results/metrics_comparison.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

## 7 · Residual Diagnostics

In [ ]:
# Change MODEL_NAME to inspect any model
MODEL_NAME = list(results.keys())[0]   # Best model by default
n = min(len(y_test_raw), len(results[MODEL_NAME]['preds']))
fig = plot_residuals(y_test_raw[:n], results[MODEL_NAME]['preds'][:n], model_name=MODEL_NAME)
if SAVE_RESULTS:
    fig.savefig(f'results/residuals_{MODEL_NAME.lower()}.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

## 8 · Feature Importance (tree models)

In [ ]:
for name, res in results.items():
    if res.get('importance') is not None:
        fig = plot_feature_importance(res['importance'], model_name=name)
        if SAVE_RESULTS:
            fig.savefig(f'results/importance_{name.lower()}.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
        plt.show()